# Spam Detection Analysis

This notebook provides exploratory data analysis and model experimentation for spam detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import sys
sys.path.append('../src')

from preprocessing import TextPreprocessor, FeatureExtractor
from models import SpamDetector, ModelComparison

plt.style.use('seaborn-v0_8')
%matplotlib inline

## Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/spam_dataset.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Basic statistics
print("Class distribution:")
print(df['label'].value_counts())
print(f"\nSpam percentage: {df['label'].value_counts()['spam'] / len(df) * 100:.1f}%")

## Text Analysis

In [ ]:
# Message length analysis
df['message_length'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Message length distribution
df.boxplot(column='message_length', by='label', ax=axes[0,0])
axes[0,0].set_title('Message Length by Class')

# Word count distribution
df.boxplot(column='word_count', by='label', ax=axes[0,1])
axes[0,1].set_title('Word Count by Class')

# Message length histogram
spam_lengths = df[df['label'] == 'spam']['message_length']
ham_lengths = df[df['label'] == 'ham']['message_length']

axes[1,0].hist(spam_lengths, alpha=0.7, label='Spam', bins=20)
axes[1,0].hist(ham_lengths, alpha=0.7, label='Ham', bins=20)
axes[1,0].set_xlabel('Message Length')
axes[1,0].set_ylabel('Frequency')
axes[1,0].legend()
axes[1,0].set_title('Message Length Distribution')

# Word count histogram
spam_words = df[df['label'] == 'spam']['word_count']
ham_words = df[df['label'] == 'ham']['word_count']

axes[1,1].hist(spam_words, alpha=0.7, label='Spam', bins=20)
axes[1,1].hist(ham_words, alpha=0.7, label='Ham', bins=20)
axes[1,1].set_xlabel('Word Count')
axes[1,1].set_ylabel('Frequency')
axes[1,1].legend()
axes[1,1].set_title('Word Count Distribution')

plt.tight_layout()
plt.show()

## Word Clouds

In [ ]:
# Create word clouds
spam_text = ' '.join(df[df['label'] == 'spam']['message'])
ham_text = ' '.join(df[df['label'] == 'ham']['message'])

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Spam word cloud
spam_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(spam_text)
axes[0].imshow(spam_wordcloud, interpolation='bilinear')
axes[0].set_title('Spam Messages Word Cloud', fontsize=16)
axes[0].axis('off')

# Ham word cloud
ham_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(ham_text)
axes[1].imshow(ham_wordcloud, interpolation='bilinear')
axes[1].set_title('Ham Messages Word Cloud', fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Model Training and Evaluation

In [ ]:
# Preprocess data
preprocessor = TextPreprocessor()
df['processed_message'] = df['message'].apply(preprocessor.preprocess)

# Extract features
feature_extractor = FeatureExtractor(method='tfidf', max_features=3000)
X = feature_extractor.fit_transform(df['processed_message'])
y = df['label']

print(f"Feature matrix shape: {X.shape}")

In [ ]:
# Train and compare models
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize model comparison
comparison = ModelComparison()

# Add models
models_to_test = ['naive_bayes', 'svm', 'random_forest', 'logistic_regression']
for model_type in models_to_test:
    detector = SpamDetector(model_type=model_type)
    comparison.add_model(model_type, detector)

# Train and evaluate
comparison.train_all(X_train, y_train)
comparison.evaluate_all(X_test, y_test)
comparison.print_comparison()

## Feature Importance Analysis

In [ ]:
# Get feature names and importance (for models that support it)
feature_names = feature_extractor.get_feature_names()

# For Naive Bayes, we can look at feature log probabilities
nb_model = comparison.models['naive_bayes'].model
feature_log_prob = nb_model.feature_log_prob_

# Get top features for spam class
spam_features = feature_log_prob[1]  # Assuming spam is class 1
top_spam_indices = np.argsort(spam_features)[-20:]  # Top 20 features

print("Top 20 features for SPAM detection:")
for i, idx in enumerate(reversed(top_spam_indices)):
    print(f"{i+1:2d}. {feature_names[idx]:20s} ({spam_features[idx]:.3f})")

## Model Performance Visualization

In [ ]:
# Create performance comparison chart
metrics_df = pd.DataFrame({
    model_name: result['metrics'] 
    for model_name, result in comparison.results.items()
}).T

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
metrics_df.plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nPerformance Summary:")
print(metrics_df.round(4))